# Radio-Sim Tutorial

This notebook demonstrates scenario-driven voice traffic and synthetic media traffic with the `radio_sim` Python bindings.

What you will do:

1. Load real voice assets from `output/comms_log.json` and `output/audio`.
2. Run a shortened scenario for quick iteration.
3. Reconstruct delivered audio and play it in-notebook.
4. Generate a synthetic mixed-media manifest (audio + video) and visualize received video frames.

## CSMA/CA Objective

In this run we use `set_csma_mac()` and observe contention behavior (collisions, queue drops, retries) under the same traffic assets.

In [ ]:
%matplotlib inline

try:
    from IPython.display import Audio, HTML, display
except Exception:  # fallback for non-notebook smoke execution
    class Audio:  # type: ignore[override]
        def __init__(self, data=None, rate=None):
            self.data = data
            self.rate = rate

    def HTML(data):
        return data

    def display(*_args, **_kwargs):
        return None

import tutorial_utils as tu

radio_sim = tu.import_radio_sim()
paths = tu.resolve_demo_paths()
HAS_VISUAL_DEPS = tu.visual_deps_available()
if not HAS_VISUAL_DEPS:
    print('Visualization dependencies (numpy/matplotlib) are missing; plot/video cells will be skipped.')
paths


## Inspect Source Assets

The `output/` folder stores the comms timeline and recorded speech clips. We summarize it first, then generate a compact subset for a short simulation.

In [ ]:
comms = tu.load_json(paths["comms_log"])
asset_summary = tu.summarize_comms_log(comms)
asset_summary

In [ ]:
compact_log_path = tu.build_compact_comms_log(
    paths["comms_log"],
    paths["generated_dir"] / "compact_comms_log.json",
    max_messages=14,
    time_scale=0.12,
    start_offset_s=0.25,
)
compact = tu.load_json(compact_log_path)
compact_summary = tu.summarize_comms_log(compact)
compact_log_path, compact_summary

## Run CSMA Voice Scenario

The full mission log spans a long wall-clock window, so we run a compact scenario with remapped start times.

In [ ]:
voice_sim, voice_summary = tu.run_voice_scenario(
    radio_sim,
    comms_log_path=compact_log_path,
    audio_dir=paths["audio_dir"],
    mac_kind="csma",
    num_nodes=28,
    sim_duration_s=24.0,
    seed=11,
)
tu.core_summary_fields(voice_summary)

In [ ]:
voice_rows = tu.voice_rows(voice_summary)
pdrs = [row["pdr"] for row in voice_rows if row.get("pdr") is not None]

if HAS_VISUAL_DEPS:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(6, 3))
    plt.plot(range(len(pdrs)), pdrs, marker="o", linewidth=1.5)
    plt.ylim(0.0, 1.05)
    plt.xlabel("Message/receiver row")
    plt.ylabel("Frame PDR")
    plt.title("Voice delivery PDR by message row")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('Skipping voice PDR plot (missing numpy/matplotlib).')

voice_rows[:8]


## Reconstruct Delivered Voice

`voice_results` includes per-frame payloads. We reconstruct one high-delivery message into PCM and play it inline.

In [ ]:
best_voice = tu.select_best_voice_result(voice_summary, min_frames=5)
if best_voice is None:
    print("No sufficiently received voice result found.")
else:
    pcm = tu.reconstruct_voice_audio(voice_sim, best_voice)
    wav_bytes = tu.pcm_to_wav_bytes(pcm, sample_rate_hz=tu.CODEC_SAMPLE_RATE_HZ)
    print(
        "message_id=", best_voice["message_id"],
        "sender=", best_voice["sender_id"],
        "receiver=", best_voice["receiver_id"],
        "frames_received=", tu.received_count(best_voice["frames_received"]),
        "pdr=", round(best_voice["pdr"], 3),
        "samples=", len(pcm) // 2,
    )
    display(Audio(data=wav_bytes))


## Run CSMA Mixed Media (Audio + Video)

`MediaScenario` uses a JSON manifest. Here we synthesize one audio stream and one video stream for a compact tutorial workload.

In [ ]:
manifest_path = tu.build_synthetic_media_manifest(
    paths["generated_dir"] / "synthetic_media_manifest.json",
    duration_s=8.0,
    audio_sender=0,
    audio_receiver=1,
    video_sender=2,
    video_receiver=3,
    audio_stream_id=8001,
    video_stream_id=8002,
    audio_payload_bytes=960,
    video_payload_bytes=4096,
)
manifest = tu.load_json(manifest_path)
manifest_path, len(manifest["frames"]), manifest["frames"][:3]

In [ ]:
if tu.media_scenario_supported(radio_sim):
    media_sim, media_summary = tu.run_media_scenario(
        radio_sim,
        manifest_path=manifest_path,
        mac_kind="csma",
        num_nodes=8,
        sim_duration_s=12.0,
        seed=17,
        mtu_bytes=1200,
        playout_slack_ms=50.0,
    )
    tu.core_summary_fields(media_summary)
else:
    media_sim = None
    media_summary = {"media_results": []}
    print(tu.media_support_hint())


In [ ]:
media_rows = tu.media_rows(media_summary)
audio_rows = tu.media_rows(media_summary, media_kind="audio")
video_rows = tu.media_rows(media_summary, media_kind="video")

print("media rows:", len(media_rows), "audio rows:", len(audio_rows), "video rows:", len(video_rows))
media_rows

In [ ]:
video_stream = tu.select_media_stream(media_summary, media_kind="video")
if video_stream is None:
    print("No video stream result found.")
elif not HAS_VISUAL_DEPS:
    print('Skipping video visualization (missing numpy/matplotlib).')
else:
    import matplotlib.pyplot as plt

    arrays = tu.media_video_arrays(video_stream, width=64, height=64, max_frames=24)
    tu.plot_video_grid(arrays[:8], title="Received video frame samples", cols=4)
    plt.show()

    anim = tu.animate_video(arrays, interval_ms=140, title="Video payload reconstruction")
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


## How To Interpret This CSMA Run

- `collisions` and `drop_events` rise when contention is high.
- `pdr_sender_confirmed` reports sender-confirmed delivery ratio.
- Per-stream `frames_late_dropped` in `media_results` captures playout deadline misses.

This notebook is meant as a practical walkthrough: change seeds, queue size, and capture margin to see contention tradeoffs.